In [2]:
import os
from pathlib import Path
from rag_pipeline import RagPipeline, crear_codificador, EMBEDDING_BACKEND, EMBEDDING_MODEL

# Desactivar temporalmente OPENAI_API_KEY para evaluar la fase pura de retrieval
os.environ.pop("OPENAI_API_KEY", None)

print("1. Verificando codificador de embeddings...")
codificador = crear_codificador()
nombre = getattr(codificador, "id_servido", getattr(codificador, "modelo", EMBEDDING_MODEL))
print(f"✓ Modelo cargado: {nombre} | Backend: {EMBEDDING_BACKEND} (Tope: {codificador.max_seq_tokens} tokens)")

print("\n2. Conectando con pipeline e indexador...")
pipeline = RagPipeline()
print(f"✓ Pipeline inicializado hacia Qdrant: {pipeline.client}")


1. Verificando codificador de embeddings...
✓ Modelo cargado: bge-m3:latest | Backend: h200 (Tope: 8192 tokens)

2. Conectando con pipeline e indexador...
✓ Pipeline inicializado hacia Qdrant: <qdrant_client.qdrant_client.QdrantClient object at 0x12230fb60>


In [3]:
# 1. Ingesta y fragmentación del corpus en tokens
print("Procesando documentos de 'corpus/'...")
chunks = pipeline.ingest(Path("corpus"), chunk_tokens=512, overlap_tokens=102)
print(f"✓ Total fragmentos generados: {len(chunks)}")

# 2. Indexación en el contenedor Qdrant (localhost:6333)
print("\nIndexando fragmentos en la colección de Qdrant...")
pipeline.index(chunks)
print("✓ Indexación completada en Qdrant.")

# 3. Consultas de prueba técnicas sobre los papers
preguntas_prueba = [
    "How does Direct Preference Optimization eliminate the need for a separate reward model in RLHF?",
    "Why does naive vector RAG struggle with global sensemaking queries according to GraphRAG?",
    "Under what conditions does Logistic Regression outperform Naive Bayes asymptotically as the training set size increases?"
]

print("\n" + "="*80)
print("RECUPERACIÓN TOP-5 DE PRUEBA")
print("="*80)

for idx, q in enumerate(preguntas_prueba, 1):
    print(f"\n--- CONSULTA {idx}: {q} ---")
    hits = pipeline.retrieve(q, top_k=5)
    for rank, h in enumerate(hits, 1):
        score = h.get("score", 0.0)
        doc = h.get("document", "desconocido")
        chunk_id = h.get("chunk_id", "")
        texto_muestra = h.get("text", "").replace("\n", " ").strip()[:110]
        print(f"[{rank}] Score: {score:.4f} | Doc: {doc} | ID: {chunk_id}")
        print(f"    Texto: {texto_muestra}...")


Procesando documentos de 'corpus/'...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (63802 > 8192). Running this sequence through the model will result in indexing errors


ingesta: brown-2020-gpt3.pdf: 75 página(s), 186342 caracteres útiles → 156 fragmento(s)
ingesta: edge-2024-graphrag.pdf: 26 página(s), 70385 caracteres útiles → 60 fragmento(s)
ingesta: ng-jordan-2001-discriminative-vs-generative.pdf: 8 página(s), 18996 caracteres útiles → 19 fragmento(s)
ingesta: rafailov-2023-dpo.pdf: 27 página(s), 71700 caracteres útiles → 66 fragmento(s)
ingesta: rag-anything-2025.pdf: 18 página(s), 57304 caracteres útiles → 41 fragmento(s)
✓ Total fragmentos generados: 342

Indexando fragmentos en la colección de Qdrant...
✓ Indexación completada en Qdrant.

RECUPERACIÓN TOP-5 DE PRUEBA

--- CONSULTA 1: How does Direct Preference Optimization eliminate the need for a separate reward model in RLHF? ---
[1] Score: 0.7057 | Doc: rafailov-2023-dpo.pdf | ID: rafailov-2023-dpo-0002
    Texto: , and then use RL to find a policy that maximizes the learned reward. In contrast, DPO directly optimizes for ...
[2] Score: 0.6949 | Doc: rafailov-2023-dpo.pdf | ID: rafailov-2023